# run_all_0416b_nb1 — fixed × fixed (anchor_high)

Part of the 0416b negotiation-data sweep (deadline-free version).

* Seller strategies covered : **anchor_high**
* Buyer strategies covered  : **all 10 fixed buyers**
* Mode                      : **fixed × fixed**
* Target per-notebook runtime : **~17 h** on `deepseek-reasoner`
* No round-limit or deadline information is included in any prompt.

The six seller strategies in the experiment universe are:
`anchor_high`, `cooperative`, `boulware_seller`, `conceder_seller`,
`tit_for_tat_matcher`, `relational`.

The ten buyer strategies are: `random`, `budget`, `wild`, `concession`,
`anchor_drag`, `boulware_buyer`, `conceder_buyer`, `tit_for_tat_buyer`,
`persistent_f0.30`, `persistent_f0.55`.


In [1]:
import os
# ── Imports ───────────────────────────────────────────────────────────
import sys
from pathlib import Path

HERE = Path.cwd()
for p in (HERE, HERE.parent, HERE.parent.parent):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from openai import OpenAI

import exp_runner_0416 as R
from run_all_0416 import run_experiment_block

from exp_runner_0416 import (
    RandomBuyer, BudgetBuyer, WildBuyer,
    ConcessionBuyer, AnchorDragBuyer,
    BoulwareBuyer, ConcederBuyer, TitForTatBuyer,
    PersistentBuyer, AdaptiveBuyer,
    fixed_seller_schedule, twostage_seller_schedule, threestage_seller_schedule,
)
from run_all_0416 import (
    all_fixed_buyers_for_notebook, fixed_pairs_for_sellers,
)


# ── Config (edit as needed) ──────────────────────────────────────────
CLIENT = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],  # Set this key in your environment.
    base_url="https://api.deepseek.com",
)
MODEL       = "deepseek-reasoner"
N_ROUNDS    = 15
BASE_SEED   = 5000 + 1000        # distinct seed offset per notebook
OUTPUT_ROOT = Path("results_0416b")
NOTEBOOK_ID = "nb1"


## EXPERIMENTS list — notebook 1

In [2]:
# One seller strategy × all 10 fixed buyers × 22 runs = 220 runs (~16.5 h).
SELLER_STRATEGY  = 'anchor_high'
N_RUNS_PER_PAIR  = 22

EXPERIMENTS = fixed_pairs_for_sellers(
    seller_strategies=[SELLER_STRATEGY],
    n_rounds=N_ROUNDS,
    n_runs_per_pair=N_RUNS_PER_PAIR,
)
print(f"Total conditions: {len(EXPERIMENTS)}")
print(f"Total runs      : {sum(e['n_runs'] for e in EXPERIMENTS)}")


Total conditions: 10
Total runs      : 220


## Run the block

In [3]:
# ── Execute ───────────────────────────────────────────────────────────
manifest = run_experiment_block(
    experiments=EXPERIMENTS,
    notebook_id=NOTEBOOK_ID,
    client=CLIENT,
    model=MODEL,
    n_rounds=N_ROUNDS,
    base_seed=BASE_SEED,
    output_root=OUTPUT_ROOT,
    verbose=True,
    resume=True,   # skip conditions whose JSON already exists
)
print("\n--- Per-condition timings (minutes) ---")
for t in manifest["timings"]:
    print(t)



[1/10]  sel_anchor_high__buy_random
  buyer   = random
  seller  = anchor_high  (adaptive=False)
  n_runs  = 22   n_rounds = 15

  --- Run 1/22  seed=6000  prod=Potato chips (50g) cost=$0.64 mkt=$1.56 int=$0.71 ---
  R 1 BUY $0.80 T2 [buy=random]  SEL $1.50 [sel=anchor_high]  "Given the market price for quality snacks like our Potato ch..."
  R 2 BUY $1.31 T5 [buy=random]  SEL $1.48 [sel=anchor_high]  "Thank you for your continued interest. While I appreciate yo..."
  R 3 BUY $1.32 T0 [buy=random]  SEL $1.47 [sel=anchor_high]  "I understand you're seeking a competitive price, but $1.32 p..."
  R 4 BUY $0.99 T5 [buy=random]  SEL $1.46 [sel=anchor_high]  "I appreciate your engagement, but $0.99 per unit is not a vi..."
  R 5 BUY $1.45 T4 [buy=random]  SEL $1.45 [sel=anchor_high]  "I appreciate you moving closer to a workable price. At $1.45..."
  R 6 BUY $1.33 T6 [buy=random]  SEL $1.44 [sel=anchor_high]  "I appreciate your commitment to the order, but $1.33 per uni..."
  R 7 BUY $1.04 

## Inspection

In [5]:
# ── Quick inspection of the first run of the first condition ────────
import json
first_tag = EXPERIMENTS[0]["tag"]
with open(OUTPUT_ROOT / NOTEBOOK_ID / f"{first_tag}.json", encoding="utf-8") as f:
    exp = json.load(f)
R.inspect_run(exp["runs"][0], max_chars=200)


  Product: Potato chips (50g)  Supplier: Direct-from-manufacturer  v=1.35  schedule=anchor_high
  Buyer: random  Model: deepseek-reasoner

  R 1 BUY $0.80 [random]  SEL $1.50 [anchor_high]
       Given the market price for quality snacks like our Potato chips (50g) is $1.56 per unit, and considering the production costs and value delivered, I can offer a competitive rate for your 33-unit order...

  R 2 BUY $1.31 [random]  SEL $1.48 [anchor_high]
       Thank you for your continued interest. While I appreciate your offer, $1.31 per unit does not align with the premium quality and production standards of our Potato chips. To support your 33-unit order...

  R 3 BUY $1.32 [random]  SEL $1.47 [anchor_high]
       I understand you're seeking a competitive price, but $1.32 per unit is not feasible given the high quality and production standards of our product. To demonstrate our commitment to your business, I ca...

  R 4 BUY $0.99 [random]  SEL $1.46 [anchor_high]
       I appreciate your 